NEW APPROACH

In [1]:
import json, re, hashlib
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd

INPUT_PATH = r"C:\Users\hpriy\OneDrive\Desktop\Fossee files\multi llm\answers.json.01.12.2025\answers.json"   # <-- change
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

TARGET_N = 100
INFRA_PHRASE = "Unable to connect to any code servers!"

# Hybrid diversity controls
PHASE1_MAX_PER_Q = 1   # strict: maximize unique questions
PHASE2_MAX_PER_Q = 2   # only if we can't reach 100 otherwise
MAX_EMPTY_CODE_ALLOWED = 0  # you can set to 1-2 if you want, but 0 makes dataset cleaner


In [2]:
def iter_json_objects(path: str):
    with open(path, "r", encoding="utf-8") as fin:
        for row_no, line in enumerate(fin, start=1):
            line = line.strip()
            if not line:
                continue
            if line.endswith(","):
                line = line[:-1]
            try:
                yield row_no, json.loads(line)
            except json.JSONDecodeError:
                continue


In [3]:
RE_EXPLICIT_SYNTAX = re.compile(
    r"("
    r"\bSyntaxError\b|"
    r"\bIndentationError\b|"
    r"\bTabError\b|"
    r"invalid syntax|"
    r"unexpected indent|"
    r"unindent does not match any outer indentation level|"
    r"expected an indented block|"
    r"EOL while scanning string literal|"
    r"EOF while parsing|"
    r"unterminated string literal|"
    r"return outside function|"
    r"break outside loop|"
    r"continue outside loop|"
    r"f-string:"
    r")",
    re.IGNORECASE
)

def explicit_syntax_type(fb: str) -> str | None:
    fb = (fb or "").strip()
    if not fb:
        return None
    if INFRA_PHRASE in fb:
        return None
    if not RE_EXPLICIT_SYNTAX.search(fb):
        return None

    if re.search(r"\bTabError\b", fb): return "SYNTAX_TabError"
    if re.search(r"\bIndentationError\b", fb): return "SYNTAX_IndentationError"
    if re.search(r"\bSyntaxError\b", fb): return "SYNTAX_SyntaxError"
    return "SYNTAX_EXPLICIT_OTHER"


In [4]:
def compile_syntax_type(code: str) -> str | None:
    code = code or ""
    try:
        compile(code, "<string>", "exec")
        return None
    except TabError:
        return "SYNTAX_TabError"
    except IndentationError:
        return "SYNTAX_IndentationError"
    except SyntaxError:
        return "SYNTAX_SyntaxError"
    except Exception:
        return None


In [5]:
def normalize_code(code: str) -> str:
    return "\n".join(ln.rstrip() for ln in (code or "").replace("\r\n", "\n").split("\n")).strip()

# reject "just function name" or "foo()"
RE_ONLY_IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
RE_ONLY_CALL  = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*\s*\(\s*\)\s*$")

def is_meaningful_code(code: str) -> bool:
    c = normalize_code(code)
    if c == "":
        return False
    if len(c) < 5:
        return False
    if RE_ONLY_IDENT.fullmatch(c):
        return False
    if RE_ONLY_CALL.fullmatch(c):
        return False

    # require structure
    return (
        ("def " in c) or ("class " in c) or
        ("return" in c) or ("if " in c) or ("for " in c) or ("while " in c) or
        ("=" in c) or (":" in c)
    )

# canonicalize: identifiers -> VAR, numbers -> NUM, strings -> STR
RE_STRING = re.compile(r"(\".*?\"|\'.*?\')", re.DOTALL)
RE_NUMBER = re.compile(r"\b\d+(\.\d+)?\b")
RE_IDENT  = re.compile(r"\b[a-zA-Z_][a-zA-Z0-9_]*\b")

PY_KEYWORDS = {
    "def","return","if","elif","else","for","while","break","continue","pass",
    "class","try","except","finally","with","as","import","from","lambda",
    "True","False","None","and","or","not","in","is","global","nonlocal","assert",
    "yield","raise"
}

def canonicalize_for_hash(code: str) -> str:
    c = normalize_code(code)
    c = RE_STRING.sub("STR", c)
    c = RE_NUMBER.sub("NUM", c)

    def repl_ident(m):
        tok = m.group(0)
        return tok if tok in PY_KEYWORDS else "VAR"
    c = RE_IDENT.sub(repl_ident, c)

    c = re.sub(r"\s+", " ", c).strip()
    return c

def canon_hash(code: str) -> str:
    canon = canonicalize_for_hash(code)
    return hashlib.sha1(canon.encode("utf-8")).hexdigest()


In [6]:
def build_syntax_100(max_per_q: int):
    picked = []
    used_q = Counter()
    seen_h = set()
    empty_used = 0

    def try_add(row_no, qid, qtext, code, fb, etype):
        nonlocal empty_used

        norm = normalize_code(code)

        if norm == "":
            if empty_used >= MAX_EMPTY_CODE_ALLOWED:
                return False
            empty_used += 1
        else:
            if not is_meaningful_code(norm):
                return False

        h = canon_hash(norm)
        if h in seen_h:
            return False

        if used_q[qid] >= max_per_q:
            return False

        used_q[qid] += 1
        seen_h.add(h)

        picked.append({
            "question": (qtext or "").strip(),
            "code": norm,
            "exec_feedback": (fb or "").strip(),
            "error_type": etype,
            "error_group": "syn",
            "question_id": qid,
            "row_no": row_no,
            "canon_hash": h,
        })
        return True

    # Pass A: explicit syntax
    for row_no, obj in iter_json_objects(INPUT_PATH):
        if len(picked) >= TARGET_N:
            break
        qid   = obj.get("question_id", "")
        qtext = obj.get("question__description") or ""
        code  = obj.get("answer") or ""
        fb    = obj.get("error") or ""

        etype = explicit_syntax_type(fb)
        if not etype:
            continue
        try_add(row_no, qid, qtext, code, fb, etype)

    # Pass B: compile fallback for empty feedback (only if still short)
    if len(picked) < TARGET_N:
        for row_no, obj in iter_json_objects(INPUT_PATH):
            if len(picked) >= TARGET_N:
                break
            qid   = obj.get("question_id", "")
            qtext = obj.get("question__description") or ""
            code  = obj.get("answer") or ""
            fb    = (obj.get("error") or "").strip()

            if fb != "":
                continue  # fallback only for empty feedback
            etype = compile_syntax_type(code)
            if not etype:
                continue
            try_add(row_no, qid, qtext, code, fb, etype)

    return picked


In [7]:
picked_phase1 = build_syntax_100(PHASE1_MAX_PER_Q)
print("Phase 1 picked:", len(picked_phase1), "| unique questions:", len(set(r["question_id"] for r in picked_phase1)))

picked = picked_phase1

if len(picked) < TARGET_N:
    print("Not enough with max_per_q=1. Expanding to allow 2 per question...")
    picked_phase2 = build_syntax_100(PHASE2_MAX_PER_Q)
    picked = picked_phase2
    print("Phase 2 picked:", len(picked), "| unique questions:", len(set(r["question_id"] for r in picked)))


Phase 1 picked: 100 | unique questions: 100


In [8]:
df = pd.DataFrame(picked)


df = df.head(TARGET_N)

out_path = OUT_DIR / "syntax_100_hybrid_unique.xlsx"
df_out = df[["question","code","exec_feedback","error_type","error_group","question_id","row_no"]]
df_out.to_excel(out_path, index=False)

print("Saved:", out_path)
print(df_out["error_type"].value_counts())
print("Unique question :", df_out["question"].nunique())
print("Unique buggy code: ", df_out["code"].nunique())
print("Rows:", len(df_out))


Saved: outputs\syntax_100_hybrid_unique.xlsx
error_type
SYNTAX_EXPLICIT_OTHER      64
SYNTAX_SyntaxError         24
SYNTAX_IndentationError    12
Name: count, dtype: int64
Unique question : 73
Unique buggy code:  100
Rows: 100
